# Kalman Filter Scenario Plots

Three scenarios using the Kalman filter implementation from `kalman_filter.ipynb`:
1. **Constant Velocity** — well-tuned baseline
2. **Mismatched R** — filter's assumed measurement noise differs from reality
3. **Quadratic Trajectory** — model mismatch, comparing baseline Q vs inflated Q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Core Kalman filter functions — from kalman_filter.ipynb

def predict(x, P, F, Q):
    x_pred = F @ x
    P_pred = F @ P @ F.T + Q
    return x_pred, P_pred

def update(x_pred, P_pred, z, H, R):
    y = z - H @ x_pred
    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T @ np.linalg.inv(S)
    x_upd = x_pred + K @ y
    I = np.eye(P_pred.shape[0])
    P_upd = (I - K @ H) @ P_pred
    return x_upd, P_upd, y, S

def kalman_filter(true_states, x, P, F, Q, H, R):
    x = x.copy()
    P = P.copy()

    true_positions = []
    measurements = []
    estimates = []
    covariances = []
    predictions = []
    prediction_covariances = []
    nis_log = []
    S_trace_log = []
    S_model_trace_log = []

    for k in range(len(true_states)):
        true_state = true_states[k]
        true_pos = H @ true_state

        noise_std = np.sqrt(R[0, 0])
        z = true_pos + np.random.normal(0, noise_std, size=2)

        x_pred, P_pred = predict(x, P, F, Q)
        predictions.append(x_pred)
        prediction_covariances.append(P_pred)
        x, P, y, S = update(x_pred, P_pred, z, H, R)
        S_model_trace_log.append(np.trace(H @ P_pred @ H.T))
        S_trace_log.append(np.trace(S))

        y = np.atleast_2d(y).reshape(-1, 1)
        nis = (y.T @ np.linalg.solve(S, y)).item()
        nis_log.append(nis)

        true_positions.append(true_pos)
        measurements.append(z)
        estimates.append([x[0], x[2]])
        covariances.append(P)

    return (
        np.array(true_positions), np.array(measurements), np.array(estimates),
        np.array(covariances), np.array(predictions), np.array(prediction_covariances),
        np.array(nis_log), np.array(S_trace_log), np.array(S_model_trace_log)
    )

def kalman_filter_mismatched_R(true_states, x, P, F, Q, H, R_sim, R_filter):
    """Run KF where noise used to generate measurements (R_sim) differs from
    what the filter assumes (R_filter)."""
    x = x.copy()
    P = P.copy()

    true_positions = []
    measurements = []
    estimates = []
    covariances = []
    nis_log = []

    for k in range(len(true_states)):
        true_state = true_states[k]
        true_pos = H @ true_state

        noise_std = np.sqrt(R_sim[0, 0])
        z = true_pos + np.random.normal(0, noise_std, size=2)

        x_pred, P_pred = predict(x, P, F, Q)
        x, P, y, S = update(x_pred, P_pred, z, H, R_filter)

        y = np.atleast_2d(y).reshape(-1, 1)
        nis = (y.T @ np.linalg.solve(S, y)).item()
        nis_log.append(nis)

        true_positions.append(true_pos)
        measurements.append(z)
        estimates.append([x[0], x[2]])
        covariances.append(P)

    return (
        np.array(true_positions), np.array(measurements),
        np.array(estimates), np.array(covariances), np.array(nis_log)
    )

In [ ]:
# Trajectory generators — from kalman_filter.ipynb

def constant_velocity_true_position(num_steps, dt, v_x, v_y):
    true_states = []
    for k in range(num_steps):
        t = k * dt
        px = v_x * t
        py = v_y * t
        true_states.append(np.array([px, v_x, py, v_y]))
    return np.array(true_states)

def circular_true_position(num_steps, dt, radius, omega):
    """Uniform circular motion: px = R·cos(ω·t), py = R·sin(ω·t).
    The CV model cannot represent sustained turning, so this is a genuine
    structural limitation rather than a tuning problem."""
    true_states = []
    t = 0.0
    for _ in range(num_steps):
        t += dt
        px =  radius * np.cos(omega * t)
        vx = -radius * omega * np.sin(omega * t)
        py =  radius * np.sin(omega * t)
        vy =  radius * omega * np.cos(omega * t)
        true_states.append(np.array([px, vx, py, vy]))
    return np.array(true_states)

In [ ]:
# Baseline parameters

dt = 1.0
num_steps = 60

# Velocities
v_x, v_y = 2.0, 1.0

# Initial state (start exactly on truth for constant-velocity scenarios)
x0 = np.array([0.0, v_x, 0.0, v_y])

# Initial covariance — moderate uncertainty
P0 = np.diag([1.0, 1.0, 1.0, 1.0])

# Process model — constant velocity
F = np.array([[1, dt, 0,  0 ],
              [0,  1, 0,  0 ],
              [0,  0, 1, dt ],
              [0,  0, 0,  1 ]])

# Process noise — sigma_acc = 0.1 m/s^2
sigma_acc = 0.1
q = sigma_acc ** 2
Q = np.array([[dt**4/4, dt**3/2,       0,       0],
              [dt**3/2,   dt**2,       0,       0],
              [      0,       0, dt**4/4, dt**3/2],
              [      0,       0, dt**3/2,   dt**2]]) * q

# Measurement matrix — observe position only
H = np.array([[1, 0, 0, 0],
              [0, 0, 1, 0]])

# Measurement noise — 1 m std
R = np.diag([1.0, 1.0])

## Scenario 1: Constant Velocity

The true trajectory follows constant velocity (`v_x=2.0, v_y=1.0` m/s). The filter model, process noise, and measurement noise are all well-matched. This is the baseline — showing the KF working as intended.

In [ ]:
np.random.seed(42)
true_states_cv = constant_velocity_true_position(num_steps, dt, v_x, v_y)

true_pos_cv, meas_cv, est_cv, cov_cv, pred_cv, pred_cov_cv, nis_cv, _, _ = kalman_filter(
    true_states_cv, x0, P0, F, Q, H, R
)

time_idx = np.arange(len(meas_cv))

# Trajectory
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(true_pos_cv[:, 0], true_pos_cv[:, 1],
        color='tab:blue', linewidth=2, label='True Position')
sc = ax.scatter(meas_cv[:, 0], meas_cv[:, 1],
                c=time_idx, cmap='viridis', s=20, alpha=0.9, label='Measurements')
ax.plot(est_cv[:, 0], est_cv[:, 1],
        color='tab:red', linewidth=2, label='KF Estimate')
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Time step')
ax.set_title('Constant Velocity — Trajectory')
ax.set_xlabel('X Position (m)')
ax.set_ylabel('Y Position (m)')
ax.grid(True)
ax.autoscale()
ax.legend()
plt.tight_layout()
plt.show()

rmse_cv = np.sqrt(np.mean(np.sum((est_cv - true_pos_cv) ** 2, axis=1)))
print(f'RMSE: {rmse_cv:.4f} m')
print(f'Mean NIS: {np.mean(nis_cv):.3f}  (expected ≈ {H.shape[0]})')

## Scenario 2: Mismatched Measurement Noise (R)

The true sensor noise is `R_actual = diag([4, 4])` (2 m std). Three filter assumptions are compared side-by-side:
- **Overconfident** `R_filter = diag([0.04, 0.04])`: filter thinks noise is 0.2 m — it trusts noisy measurements far too much
- **Matched** `R_filter = diag([4, 4])`: correctly specified
- **Underconfident** `R_filter = diag([400, 400])`: filter thinks noise is 20 m — it largely ignores measurements and coasts on the motion model

In [ ]:
R_actual       = np.diag([4.0,   4.0  ])
R_overconfident = np.diag([0.04,  0.04 ])
R_matched       = np.diag([4.0,   4.0  ])
R_underconfident = np.diag([400.0, 400.0])

cases_r = [
    ('Overconfident\n($R_{filter}$ = 0.04·I)',   R_actual, R_overconfident),
    ('Matched\n($R_{filter}$ = $R_{actual}$)',    R_actual, R_matched),
    ('Underconfident\n($R_{filter}$ = 400·I)',    R_actual, R_underconfident),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, (title, R_sim, R_filt) in zip(axes, cases_r):
    np.random.seed(42)
    true_pos_r, meas_r, est_r, _, _ = kalman_filter_mismatched_R(
        true_states_cv, x0, P0, F, Q, H, R_sim, R_filt
    )
    time_idx_r = np.arange(len(meas_r))
    ax.plot(true_pos_r[:, 0], true_pos_r[:, 1],
            color='tab:blue', linewidth=2, label='True Position')
    sc = ax.scatter(meas_r[:, 0], meas_r[:, 1],
                    c=time_idx_r, cmap='viridis', s=20, alpha=0.9, label='Measurements')
    ax.plot(est_r[:, 0], est_r[:, 1],
            color='tab:red', linewidth=2, label='KF Estimate')
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label('Time step')
    ax.set_title(title)
    ax.set_xlabel('X Position (m)')
    ax.set_ylabel('Y Position (m)')
    ax.grid(True)
    ax.autoscale()
    ax.legend()

plt.suptitle('Mismatched Measurement Noise (R) — Constant Velocity Trajectory', fontsize=13)
plt.tight_layout()
plt.show()

## Scenario 3: Circular Trajectory — Baseline Q vs Inflated Q

The true trajectory follows uniform circular motion ($p_x = R\cos\omega t,\; p_y = R\sin\omega t$), but the filter uses a constant-velocity model. A CV model cannot represent sustained turning — the direction change is continuous and structural, not just a tuning problem.

Two Q settings are compared:

- **Baseline Q** (`σ_acc = 0.1`): the filter is honest about its motion model uncertainty — the estimate visibly lags and drifts away from the circle
- **Inflated Q** (`σ_acc = 3.0`): large process uncertainty forces the filter to weight measurements heavily, so the trajectory *looks* well-tracked — but the NIS plot reveals the filter's uncertainty bookkeeping is wrong, motivating the chi-squared consistency test

In [ ]:
# Circular trajectory parameters
# omega=0.2 rad/s → centripetal acc = 0.8 m/s², 8x sigma_acc — filter clearly can't keep up
# 25 steps covers ~0.8 of a circle, enough to see lag develop without looping back
num_steps_circ = 25
radius = 20.0
omega  = 0.2

# Initial state: position at t=0 on the circle, moving tangentially
x0_circ = np.array([radius, 0.0, 0.0, radius * omega])

true_states_circ = circular_true_position(num_steps_circ, dt, radius, omega)

# Inflated Q — sigma_acc = 3.0
sigma_acc_inflated = 3.0
q_infl = sigma_acc_inflated ** 2
Q_inflated = np.array([[dt**4/4, dt**3/2,       0,       0],
                        [dt**3/2,   dt**2,       0,       0],
                        [      0,       0, dt**4/4, dt**3/2],
                        [      0,       0, dt**3/2,   dt**2]]) * q_infl

# Run baseline Q
np.random.seed(42)
true_pos_cb, meas_cb, est_cb, _, _, _, nis_cb, _, _ = kalman_filter(
    true_states_circ, x0_circ, P0, F, Q, H, R
)

# Run inflated Q (same seed so same measurements)
np.random.seed(42)
true_pos_ci, meas_ci, est_ci, _, _, _, nis_ci, _, _ = kalman_filter(
    true_states_circ, x0_circ, P0, F, Q_inflated, H, R
)

print(f'Baseline Q  — Mean NIS: {np.mean(nis_cb):.3f}')
print(f'Inflated Q  — Mean NIS: {np.mean(nis_ci):.3f}')
print(f'Expected NIS ≈ {H.shape[0]}')

In [ ]:
# 2x2 layout: left column = trajectory, right column = position error
# Top row = baseline Q, bottom row = inflated Q
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

time_idx_c = np.arange(num_steps_circ)

cases = [
    (est_cb, meas_cb, true_pos_cb, nis_cb, f'Baseline Q  (σ_acc = {sigma_acc})',           axes[0]),
    (est_ci, meas_ci, true_pos_ci, nis_ci, f'Inflated Q  (σ_acc = {sigma_acc_inflated})',  axes[1]),
]

for (est, meas, true_pos, nis, label, row_axes) in cases:
    ax_traj, ax_err = row_axes

    # --- Trajectory ---
    ax_traj.scatter(meas[:, 0], meas[:, 1],
                    color='grey', s=12, alpha=0.4, label='Measurements', zorder=1)
    ax_traj.plot(true_pos[:, 0], true_pos[:, 1],
                 color='tab:blue', linewidth=2, label='True Position', zorder=2)
    ax_traj.plot(est[:, 0], est[:, 1],
                 color='tab:red', linewidth=2, label='KF Estimate', zorder=3)
    ax_traj.scatter(true_pos[0, 0], true_pos[0, 1],
                    color='green', s=120, zorder=5, label='Start')

    mid = len(true_pos) // 4
    ax_traj.annotate('',
                     xy=true_pos[mid + 1],
                     xytext=true_pos[mid],
                     arrowprops=dict(arrowstyle='->', color='tab:blue', lw=2))

    ax_traj.set_aspect('equal')
    ax_traj.set_title(f'{label} — Trajectory')
    ax_traj.set_xlabel('X Position (m)')
    ax_traj.set_ylabel('Y Position (m)')
    ax_traj.grid(True)
    ax_traj.legend()

    # --- Position error ---
    error = np.linalg.norm(est - true_pos, axis=1)
    ax_err.plot(time_idx_c, error, color='tab:red', linewidth=2)
    ax_err.set_title(f'{label} — Position Error')
    ax_err.set_xlabel('Time step')
    ax_err.set_ylabel('Position Error (m)')
    ax_err.grid(True)

plt.suptitle('Circular Trajectory — Constant-Velocity Filter', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# NIS plots — without reference line
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

time_idx_c = np.arange(num_steps_circ)

for ax, (nis, title) in zip(axes, [
    (nis_cb, f'NIS — Baseline Q  (σ_acc = {sigma_acc})'),
    (nis_ci, f'NIS — Inflated Q  (σ_acc = {sigma_acc_inflated})'),
]):
    ax.plot(time_idx_c, nis, color='tab:blue', linewidth=1.5, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Time step')
    ax.set_ylabel('NIS')
    ax.grid(True)

plt.suptitle('Normalised Innovation Squared — Circular Trajectory', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# NIS plots — with chi-squared reference line at y = 2 (expected NIS for 2D measurements)
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

time_idx_c = np.arange(num_steps_circ)

for ax, (nis, title) in zip(axes, [
    (nis_cb, f'NIS — Baseline Q  (σ_acc = {sigma_acc})'),
    (nis_ci, f'NIS — Inflated Q  (σ_acc = {sigma_acc_inflated})'),
]):
    ax.plot(time_idx_c, nis, color='tab:blue', linewidth=1.5, alpha=0.8, label='NIS')
    ax.axhline(y=2, color='black', linestyle='--', linewidth=1.5, label='Expected NIS = 2')
    ax.set_title(title)
    ax.set_xlabel('Time step')
    ax.set_ylabel('NIS')
    ax.grid(True)
    ax.legend()

plt.suptitle('Normalised Innovation Squared — Circular Trajectory', fontsize=13)
plt.tight_layout()
plt.show()